# Phase 1 Starter — Train & Deploy (Individual, 35%)
This is Session 2's demo generalized: point it at **your** dataset and work top to bottom.
Milestones: S1 pick dataset → S2 first training run → S3 tuned + endpoint attempt →
**working endpoint by start of S4** → report (template in `project/`) due S5.

In [20]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'src').exists():
    p = p.parent
sys.path.insert(0, str(p / 'src')); REPO = p
import warnings; warnings.filterwarnings('ignore')

In [21]:
import sys, pathlib
REPO = pathlib.Path(r"e:\AUBG\5th semester\ML\ml-project")
sys.path.insert(0, str(REPO / 'src'))
import warnings; warnings.filterwarnings('ignore')

from course_utils import data
DATASET = 'credit_default'        # <- your pick from project/datasets.md
TARGET = 'default payment next month'                    # <- your label column
df_raw = data.load_raw(DATASET)  
import xgboost; print('xgboost', xgboost.__version__)
df_raw.head()



xgboost 2.1.4


,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


## 1. Profile & clean (Session 1 toolkit) — document ≥2 decisions for the report

In [22]:
data.profile_frame(df_raw) #only reading the data
# your cleaning here. Watch for target leakage: is any feature only known AFTER the outcome?

shape: 30,000 rows x 25 cols | duplicated rows: 0


,dtype,missing,missing_%,unique,sample
ID,int64,0,0.0,30000,1
LIMIT_BAL,int64,0,0.0,81,20000
SEX,int64,0,0.0,2,2
EDUCATION,int64,0,0.0,7,2
MARRIAGE,int64,0,0.0,4,1
AGE,int64,0,0.0,56,24
PAY_0,int64,0,0.0,11,2
PAY_2,int64,0,0.0,11,2
PAY_3,int64,0,0.0,11,-1
PAY_4,int64,0,0.0,11,-1


In [23]:
display(df_raw.describe().T[['min', 'max', 'mean']])

,min,max,mean
ID,1.0,30000.0,15000.500000
LIMIT_BAL,10000.0,1000000.0,167484.322667
SEX,1.0,2.0,1.603733
EDUCATION,0.0,6.0,1.853133
MARRIAGE,0.0,3.0,1.551867
AGE,21.0,79.0,35.485500
PAY_0,-2.0,8.0,-0.016700
PAY_2,-2.0,8.0,-0.133767
PAY_3,-2.0,8.0,-0.166200
PAY_4,-2.0,8.0,-0.220667


In [24]:
#DUPLICATES
print('Duplicate rows (with ID):', df_raw.duplicated().sum())
print('Duplicate rows (without ID):', df_raw.drop(columns=['ID']).duplicated().sum())


bill = [f'BILL_AMT{i}' for i in range(1, 7)]
pay_amt = [f'PAY_AMT{i}' for i in range(1, 7)]

dups = df_raw[df_raw.drop(columns=['ID']).duplicated(keep=False)]
zero = dups[bill + pay_amt].abs().sum(axis=1) == 0 #checks for 0 activity
print('pairs:', len(dups) // 2, '| zero-activity:', zero.sum() // 2, '| non-zero:', (~zero).sum() // 2)
display(dups[~zero].sort_values(['LIMIT_BAL', 'AGE']))

Duplicate rows (with ID): 0
Duplicate rows (without ID): 35
pairs: 35 | zero-activity: 30 | non-zero: 5


,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
12430,12431,20000,1,2,2,24,2,2,4,4,...,1650,1650,1650,0,0,0,0,0,0,1
14294,14295,20000,1,2,2,24,2,2,4,4,...,1650,1650,1650,0,0,0,0,0,0,1
1211,1212,160000,2,3,2,26,-1,-1,-1,-1,...,390,390,390,390,390,390,390,390,390,0
21881,21882,160000,2,3,2,26,-1,-1,-1,-1,...,390,390,390,390,390,390,390,390,390,0
11741,11742,180000,1,2,1,26,-1,-1,-1,-1,...,396,396,396,396,396,396,396,396,396,0
29265,29266,180000,1,2,1,26,-1,-1,-1,-1,...,396,396,396,396,396,396,396,396,396,0
9644,9645,210000,1,2,1,39,-1,-1,-1,-1,...,1443,1443,1443,1443,1443,1443,1443,1443,1443,0
27351,27352,210000,1,2,1,39,-1,-1,-1,-1,...,1443,1443,1443,1443,1443,1443,1443,1443,1443,0
627,628,230000,1,1,1,39,-1,-1,-1,-1,...,660,660,660,660,660,660,660,660,660,0
20875,20876,230000,1,1,1,39,-1,-1,-1,-1,...,660,660,660,660,660,660,660,660,660,0


In [25]:
#Data analysis
print('AGE range:', df_raw['AGE'].min(), '-', df_raw['AGE'].max())
print('LIMIT_BAL <= 0:', (df_raw['LIMIT_BAL'] <= 0).sum())
print('rows with negative PAY_AMT:', (df_raw[pay_amt] < 0).any(axis=1).sum())
print('rows with negative BILL_AMT:', (df_raw[bill] < 0).any(axis=1).sum())
print('EDUCATION 0/5/6: %.1f%%' % (df_raw['EDUCATION'].isin([0, 5, 6]).mean() * 100))
print('MARRIAGE 0: %.1f%%' % ((df_raw['MARRIAGE'] == 0).mean() * 100))
print('default rate: %.1f%%' % (df_raw[TARGET].mean() * 100))

AGE range: 21 - 79
LIMIT_BAL <= 0: 0
rows with negative PAY_AMT: 0
rows with negative BILL_AMT: 1930
EDUCATION 0/5/6: 1.1%
MARRIAGE 0: 0.2%
default rate: 22.1%


In [26]:
display(df_raw.groupby('PAY_0')[TARGET].mean().round(2))
#Exploratory analysis:

PAY_0
-2    0.13
-1    0.17
 0    0.13
 1    0.34
 2    0.69
 3    0.76
 4    0.68
 5    0.50
 6    0.55
 7    0.78
 8    0.58
Name: default payment next month, dtype: float64

In [27]:
print(df_raw['EDUCATION'].value_counts().sort_index(), '\n')
print(df_raw['MARRIAGE'].value_counts().sort_index(), '\n')

EDUCATION
0       14
1    10585
2    14030
3     4917
4      123
5      280
6       51
Name: count, dtype: int64 

MARRIAGE
0       54
1    13659
2    15964
3      323
Name: count, dtype: int64 



In [28]:
#Reconstruct
df = df_raw.copy()
df = df.drop(columns=['ID'])
df['EDUCATION'] = df['EDUCATION'].replace({0: 4, 5: 4, 6: 4})
df['MARRIAGE'] = df['MARRIAGE'].replace({0: 3})
df['EDUCATION'] = df['EDUCATION'].map({1: 'graduate', 2: 'university', 3: 'high_school', 4: 'other'})
df['MARRIAGE'] = df['MARRIAGE'].map({1: 'married', 2: 'single', 3: 'other'})
df = df.drop(columns=['SEX'])


## 2. Features & splits

In [29]:
import pandas as pd
y = (df[TARGET] == 'yes').astype(int) if df[TARGET].dtype == object else df[TARGET]
X = pd.get_dummies(df.drop(columns=[TARGET]), drop_first=True)
S = data.make_splits(X, y)
print({k: v.shape for k, v in S.items() if hasattr(v, 'shape')})

{'X_train': (21000, 25), 'y_train': (21000,), 'X_val': (4500, 25), 'y_val': (4500,), 'X_test': (4500, 25), 'y_test': (4500,)}


## 3. Baseline → tuned model (≥2 hyperparameters, show the effect)

In [30]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from course_utils import viz

# 1. Always predict the majority (no default)
dummy = DummyClassifier(strategy='most_frequent').fit(S['X_train'], S['y_train'])
viz.metric_report(S['y_val'], dummy.predict_proba(S['X_val'])[:, 1], name='majority (val)')

# 2. Logistic regression (features scaled first)
logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(S['X_train'], S['y_train'])
viz.metric_report(S['y_val'], logreg.predict_proba(S['X_val'])[:, 1], name='logistic regression (val)')

# 3. XGBoost
model = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1,
                      eval_metric='auc', random_state=42).fit(S['X_train'], S['y_train'])
viz.metric_report(S['y_val'], model.predict_proba(S['X_val'])[:, 1], name='XGBoost (val)')

--- majority (val) ---
  accuracy: 0.779
 precision: 0.000
    recall: 0.000
        f1: 0.000
   auc_roc: 0.500
--- logistic regression (val) ---
  accuracy: 0.809
 precision: 0.697
    recall: 0.243
        f1: 0.361
   auc_roc: 0.714
--- XGBoost (val) ---
  accuracy: 0.818
 precision: 0.669
    recall: 0.354
        f1: 0.463
   auc_roc: 0.774


{'accuracy': 0.8184444444444444,
 'precision': 0.6692015209125475,
 'recall': 0.35376884422110555,
 'f1': 0.462853385930309,
 'auc_roc': 0.7740696349077771}

| Model | Accuracy | Precision | Recall | F1 | AUC (val) |
|---|---|---|---|---|---|
| Majority baseline | 0.779 | 0.000 | 0.000 | 0.000 | 0.500 |
| Logistic regression | 0.809 | 0.697 | 0.243 | 0.361 | 0.714 |
| XGBoost | 0.818 | 0.669 | 0.354 | 0.463 | 0.774 |

In [31]:
#tuning
from sklearn.metrics import roc_auc_score

def run(**params):
    base = dict(n_estimators=200, max_depth=4, learning_rate=0.1,
                eval_metric='auc', random_state=42)
    base.update(params)
    m = XGBClassifier(**base).fit(S['X_train'], S['y_train'])
    return (roc_auc_score(S['y_train'], m.predict_proba(S['X_train'])[:, 1]),
            roc_auc_score(S['y_val'],   m.predict_proba(S['X_val'])[:, 1]))

# learning_rate stays 0.1
print('--- max_depth ---')
for d in [2, 4, 8, 12]:
    tr, va = run(max_depth=d)
    print(f"max_depth={d}: train AUC={tr:.3f}, val AUC={va:.3f}, gap={tr-va:.3f}")

# max_depth is 2
print('--- learning_rate ---')
for lr in [0.01, 0.1, 0.3]:
    tr, va = run(learning_rate=lr, max_depth=2)
    print(f"learning_rate={lr}: train AUC={tr:.3f}, val AUC={va:.3f}, gap={tr-va:.3f}")

--- max_depth ---
max_depth=2: train AUC=0.801, val AUC=0.777, gap=0.024
max_depth=4: train AUC=0.851, val AUC=0.774, gap=0.077
max_depth=8: train AUC=0.982, val AUC=0.762, gap=0.220
max_depth=12: train AUC=1.000, val AUC=0.755, gap=0.245
--- learning_rate ---
learning_rate=0.01: train AUC=0.774, val AUC=0.760, gap=0.014
learning_rate=0.1: train AUC=0.801, val AUC=0.777, gap=0.024
learning_rate=0.3: train AUC=0.822, val AUC=0.775, gap=0.047


| Hyperparameter | Value | Train AUC | Val AUC | Gap |
|---|---|---|---|---|
| max_depth | 2 | 0.801 | **0.777** | 0.024 |
| max_depth | 4 | 0.851 | 0.774 | 0.077 |
| max_depth | 8 | 0.982 | 0.762 | 0.220 |
| max_depth | 12 | 1.000 | 0.755 | 0.245 |
| learning_rate | 0.01 | 0.774 | 0.760 | 0.014 |
| learning_rate | **0.1** | 0.801 | **0.777** | 0.024 |
| learning_rate | 0.3 | 0.822 | 0.775 | 0.047 |

In [32]:
# Final model
model = XGBClassifier(n_estimators=200, max_depth=2, learning_rate=0.1,
                    eval_metric='auc', random_state=42).fit(S['X_train'], S['y_train'])
viz.metric_report(S['y_val'], model.predict_proba(S['X_val'])[:, 1], name='XGBoost tuned (val)')

--- XGBoost tuned (val) ---
  accuracy: 0.817
 precision: 0.663
    recall: 0.354
        f1: 0.461
   auc_roc: 0.777


{'accuracy': 0.8173333333333334,
 'precision': 0.6629001883239172,
 'recall': 0.35376884422110555,
 'f1': 0.4613368283093054,
 'auc_roc': 0.7769625588713897}

The model catches only 35.4% of defaulters - extremely low for a credit risk prediction. Lower thresholds are tested as shown below:

In [33]:
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

p_val = model.predict_proba(S['X_val'])[:, 1]
rows = []
for t in [0.2, 0.25, 0.3, 0.35, 0.4, 0.5]:
    pred = p_val >= t
    rows.append({'threshold': t,
                 'precision': precision_score(S['y_val'], pred),
                 'recall': recall_score(S['y_val'], pred),
                 'f1': f1_score(S['y_val'], pred),
                 'flagged_%': pred.mean() * 100})
res = pd.DataFrame(rows).round(3)
display(res)

best_t = res.loc[res['f1'].idxmax(), 'threshold']
print(f'threshold {best_t} (chosen):')
print(confusion_matrix(S['y_val'], p_val >= best_t))
print(f'threshold 0.5 (default):')
print(confusion_matrix(S['y_val'], p_val >= 0.5))

,threshold,precision,recall,f1,flagged_%
0,0.20,0.440,0.658,0.527,33.111
1,0.25,0.511,0.567,0.537,24.533
2,0.30,0.560,0.508,0.533,20.022
3,0.35,0.595,0.441,0.507,16.400
4,0.40,0.635,0.407,0.496,14.178
5,0.50,0.663,0.354,0.461,11.800


threshold 0.25 (chosen):
[[2965  540]
 [ 431  564]]
threshold 0.5 (default):
[[3326  179]
 [ 643  352]]


Chosen threshold: 0.25 (highest F1 on validation). 

In [34]:
#final model input
print('max_depth:', model.get_params()['max_depth'], '| learning_rate:', model.get_params()['learning_rate'], '| threshold:', best_t)

max_depth: 2 | learning_rate: 0.1 | threshold: 0.25


## 4. Final honest number — test set, touched ONCE

In [35]:
final = viz.metric_report(S['y_test'], model.predict_proba(S['X_test'])[:, 1],threshold=best_t, name=f'FINAL (test, threshold={best_t})')
from course_utils import models
models.save_model(model, f'phase1_{DATASET}', metrics=final)

--- FINAL (test, threshold=0.25) ---
  accuracy: 0.784
 precision: 0.510
    recall: 0.593
        f1: 0.549
   auc_roc: 0.779
saved -> E:\AUBG\5th semester\ML\ml-project\models\phase1_credit_default.joblib


WindowsPath('E:/AUBG/5th semester/ML/ml-project/models/phase1_credit_default.joblib')

## 5. Deploy
```python
from course_utils import aws
aws.deploy_sklearn_model(f'models/phase1_{DATASET}.joblib', endpoint_name=f'phase1-{DATASET}-YOURNAME')
# invoke it, screenshot it for the report, then DELETE it:
# aws.delete_endpoint(f'phase1-{DATASET}-YOURNAME')
```
Sandbox blocking you? Local FastAPI twin accepted with prior approval — see `session4/local_api.py`.

In [36]:
# Workaround: on a laptop, sagemaker.get_execution_role() can't detect a role,
# so return the SageMakerCourseRole ARN saved in COURSE_SAGEMAKER_ROLE.
import os, sagemaker
sagemaker.get_execution_role = lambda *args, **kwargs: os.environ['COURSE_SAGEMAKER_ROLE']

In [37]:
# Workaround: match the container to the training environment.
# xgboost pinned to 2.1.4 (the model is trained with it; 3.x stores base_score in a format 2.1.4 can't read).
# pandas 1.5.3 is new enough for xgboost (Float32Dtype) and compatible with the container's numpy 1.24.
from course_utils import aws
aws.SERVING_REQUIREMENTS = 'xgboost==2.1.4\nscikit-learn>=1.3\npandas==1.5.3\nnumpy<2\n'
print(repr(aws.SERVING_REQUIREMENTS))

'xgboost==2.1.4\nscikit-learn>=1.3\npandas==1.5.3\nnumpy<2\n'


In [19]:
from course_utils import aws

model_path = str(REPO / 'models' / f'phase1_{DATASET}.joblib')
endpoint_name = 'phase1-credit-default-plamena-v4'

result = aws.deploy_sklearn_model(model_path, endpoint_name=endpoint_name)
print(result)

SKLearnModel is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelBuilder` (`from sagemaker.serve import ModelBuilder`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


-------!

SKLearnPredictor is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `sagemaker.core.resources.Endpoint`.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


endpoint 'phase1-credit-default-plamena-v4' is deploying - this bills hourly; delete after class!
  (first start takes ~2 min longer than you expect: the container pip-installs xgboost)
SKLearnPredictor: {'endpoint_name': 'phase1-credit-default-plamena-v4', 'sagemaker_session': <sagemaker.session.Session object at 0x000001A567B8F5C0>, 'serializer': <sagemaker.base_serializers.NumpySerializer object at 0x000001A5651587D0>, 'deserializer': <sagemaker.base_deserializers.NumpyDeserializer object at 0x000001A565158050>}


In [40]:
from sagemaker.sklearn import SKLearnPredictor
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer
import json, joblib

result = SKLearnPredictor(endpoint_name)          # connect to the deployed endpoint
result.serializer = JSONSerializer()              # inference.py expects JSON with column names
result.deserializer = JSONDeserializer()

row = S['X_test'].iloc[[0]]                       # one client from the test set
payload = {"instances": json.loads(row.to_json(orient='records'))}

out = result.predict(payload)                     # call the SageMaker endpoint
p_endpoint = out['churn_probability'][0]          # key name comes from inference.py; = probability of default
print('Endpoint response:', out)
print(f'Probability of default: {p_endpoint:.3f} -> flagged: {p_endpoint >= best_t} (threshold {best_t})')

local = joblib.load(model_path)                   # same model locally, as a sanity check
print(f'Local model probability: {local.predict_proba(row)[:, 1][0]:.3f}')
print('Actual outcome:', S['y_test'].iloc[0])

Endpoint response: {'churn_probability': [0.103]}
Probability of default: 0.103 -> flagged: False (threshold 0.25)
Local model probability: 0.103
Actual outcome: 0


In [41]:
aws.delete_endpoint(endpoint_name)

deleted endpoint + config: phase1-credit-default-plamena-v4
